In [2]:
#####################################################
# Imports and Simulation IO
#####################################################
import os 
from MDToolkit.data.objects import Topology, Frame, Simulation, LAMMPS_CustomDump_Reader
from MDToolkit.data.misc_objects import BoxVolume
from MDToolkit.utils.structure_file_utils import create_elements_dictionary
from MDToolkit.analysis.density import axial_density, axial_density_time_averaged

filedir = "/media/jrjoseph/Elements/projects/training/graphene_water_kcl_ls6/"
filename = "graphene_water_kcl_5V_prod1.out"

filepath = os.path.join(filedir, filename)

type_mapping = {
    1 : "Cl",
    2 : "H",
    3 : "O",
    4 : "C",
    5 : "K",
}

charges_dict = {
    1 : -1.0,
    2 : 0.4238,
    3 : -0.8476,
    4 : 0.0,
    5 : 1.0
}

topol = Topology(type_mapping, elements_dict = create_elements_dictionary(), charges_dict = charges_dict)

reader = LAMMPS_CustomDump_Reader(filepath, topol)

simulation = Simulation(filepath, topol, LAMMPS_CustomDump_Reader)

vols = [BoxVolume([-52.074, -25.5648857972, -27.155], [-1.0, 25.5648857972, 27.155]), BoxVolume([1.0, -25.5648857972, -27.155], [52.091, 25.5648857972, 27.155])]

In [ ]:
#####################################################
# Plotting Functions
#####################################################

import matplotlib.pyplot as plt
import numpy as np

# Axial Density

def plot_axial_number_density(
    density_data,
    show_std=True,
    figsize=(8, 5),
    ax=None
):
    """
    Plot the axial number density profile.
    """

    if ax is None:
        _, ax = plt.subplots(figsize=figsize)

    z = density_data["bin_centers"]
    mean = density_data["number_density"]["mean"]

    ax.plot(z, mean, lw=2, label="Number density")

    if show_std:
        std = density_data["number_density"]["std"]
        ax.fill_between(
            z,
            mean - std,
            mean + std,
            alpha=0.3
        )

    ax.set_xlabel(r"Position ($\AA$)")
    ax.set_ylabel(r"Number Density (atoms/$\AA^3$)")
    ax.set_title("Axial Number Density")
    ax.legend()
    ax.grid(alpha=0.3)

    return ax

def plot_axial_mass_density(
    density_data,
    show_std=True,
    figsize=(8, 5),
    ax=None
):
    """
    Plot the axial mass density profile.
    """

    if ax is None:
        _, ax = plt.subplots(figsize=figsize)

    z = density_data["bin_centers"]
    mean = density_data["mass_density"]["mean"]

    ax.plot(z, mean, lw=2, label="Mass density")

    if show_std:
        std = density_data["mass_density"]["std"]
        ax.fill_between(
            z,
            mean - std,
            mean + std,
            alpha=0.3
        )

    median = np.nanmedian(mean)

    ax.axhline(
        median,
        ls="--",
        lw=2,
        label=f"Median = {median:.3f} g/cm$^3$"
    )

    ax.set_xlabel(r"Position ($\AA$)")
    ax.set_ylabel(r"Mass Density (g/cm$^3$)")
    ax.set_title("Axial Mass Density")
    ax.legend()
    ax.grid(alpha=0.3)

    return ax

def plot_axial_elemental_number_density(
    density_data,
    topology,
    atom_types=None,
    show_std=True,
    figsize=(8, 5),
    ax=None
):
    """
    Plot elemental axial number density profiles.

    Parameters
    ----------
    topology : Topology
        Used to convert atom types into element symbols.

    atom_types : sequence[int] or None
        Types to plot. If None, all available types are plotted.
    """

    if ax is None:
        _, ax = plt.subplots(figsize=figsize)

    z = density_data["bin_centers"]

    elemental = density_data["elemental_number_density"]

    if atom_types is None:
        atom_types = sorted(elemental.keys())

    for atom_type in atom_types:

        if atom_type not in elemental:
            continue

        label = topology.type_mapping[atom_type]

        mean = elemental[atom_type]["mean"]

        ax.plot(
            z,
            mean,
            lw=2,
            label=label
        )

        if show_std:
            std = elemental[atom_type]["std"]

            ax.fill_between(
                z,
                mean - std,
                mean + std,
                alpha=0.2
            )

    ax.set_xlabel(r"Position ($\AA$)")
    ax.set_ylabel(r"Number Density (atoms/$\AA^3$)")
    ax.set_title("Elemental Axial Number Density")
    ax.legend()
    ax.grid(alpha=0.3)

    return ax

# Radial Density

In [ ]:
density_axial = axial_density_time_averaged(simulation, volumes = vols, bin_width = 0.25)

  0%|          | 0/2001 [00:00<?, ?it/s]

In [ ]:
plot_axial_number_density(density_axial)
plt.show()
plot_axial_mass_density(density_axial)
plt.show()
plot_axial_elemental_number_density(density_axial,topol)
plt.show()